In [1]:
mkdir -p data/raw

In [2]:
!curl -L -o data/raw/yellow_tripdata_2024-10.parquet \
  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 61.3M  100 61.3M    0     0  8870k      0  0:00:07  0:00:07 --:--:-- 10.1M


In [3]:
!curl -L -o data/raw/yellow_tripdata_2024-10.parquet https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 61.3M  100 61.3M    0     0  10.5M      0  0:00:05  0:00:05 --:--:-- 11.4M


In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder\
                    .appName("spark-seminar-03")\
                    .master("local[*]")\
                    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 12:23:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/30 12:23:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
spark

In [6]:
trips = spark.read.parquet("file:///materials/seminar_03_spark/data/raw/yellow_tripdata_2024-10.parquet")

In [7]:
trips.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [9]:
trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [10]:
trips.select("VendorID", "tpep_pickup_datetime", "fare_amount")

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, fare_amount: double]

In [11]:
trips

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double]

In [12]:
trips.select("VendorID", "tpep_pickup_datetime", "fare_amount").show(5)

+--------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|fare_amount|
+--------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|       18.4|
|       1| 2024-10-01 00:12:20|       14.2|
|       1| 2024-10-01 00:04:46|       13.5|
|       1| 2024-10-01 00:12:10|       14.2|
|       1| 2024-10-01 00:30:22|        3.0|
+--------+--------------------+-----------+
only showing top 5 rows



In [13]:
trips.count()

3833771

In [ ]:
SELECT 
    COUNT(*) AS trips_count,
    MIN(tpep_pickup_datetime) AS min_pickup,
    MAX(tpep_pickup_datetime) AS max_pickup
FROM trips;

In [10]:
from pyspark.sql.functions import (max, min, avg, sum, count, col)
trips.agg(
    count("*").alias("trips_count"),
    min("tpep_pickup_datetime").alias("min_pickup"),
    max("tpep_pickup_datetime").alias("max_pickup"),
).show(5)

+-----------+-------------------+-------------------+
|trips_count|         min_pickup|         max_pickup|
+-----------+-------------------+-------------------+
|    3833771|2009-01-01 00:35:59|2024-11-14 18:30:00|
+-----------+-------------------+-------------------+



In [ ]:
SELECT *,
DATE(pickup_datetime) AS pickup_date
FROM trips;

In [12]:
from pyspark.sql.functions import to_date

In [22]:
trips_new_col = trips.withColumn(
    "pickup_date", to_date(col("tpep_pickup_datetime"))
)
trips_new_col.select("VendorID", "tpep_pickup_datetime", "pickup_date").show(5)

+--------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|pickup_date|
+--------+--------------------+-----------+
|       2| 2024-10-01 00:30:44| 2024-10-01|
|       1| 2024-10-01 00:12:20| 2024-10-01|
|       1| 2024-10-01 00:04:46| 2024-10-01|
|       1| 2024-10-01 00:12:10| 2024-10-01|
|       1| 2024-10-01 00:30:22| 2024-10-01|
+--------+--------------------+-----------+
only showing top 5 rows



In [24]:
bad_trips = trips.filter(
    (col("trip_distance") <= 0) |
    (col("total_amount") <= 0)
)
bad_trips.count()

138452

In [25]:
clean_trips = trips.filter(
    (col("trip_distance") > 0) |
    (col("total_amount") > 0)
)

In [26]:
clean_trips.count()

3828726

In [28]:
clean_trips.explain(True)

== Parsed Logical Plan ==
'Filter (('trip_distance > 0) OR ('total_amount > 0))
+- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] parquet

== Analyzed Logical Plan ==
VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double
Filter ((trip_distance#4 > cast(0 as double)) OR (total_amount#16 > cast(0 as double)))
+- Relation [Vendo

In [13]:
clean_trips = trips.withColumn(
    "pickup_date", to_date(col("tpep_pickup_datetime"))
).filter(
    (col("trip_distance") > 0) |
    (col("total_amount") > 0)
)

In [37]:
trips_by_day = clean_trips.groupBy("pickup_date")\
                         .agg(count("*").alias("trips_count"))\
                         .orderBy("pickup_date")

In [38]:
trips_by_day.show(5)

[Stage 26:==============================================>         (10 + 2) / 12]

+-----------+-----------+
|pickup_date|trips_count|
+-----------+-----------+
| 2009-01-01|          1|
| 2024-09-30|         12|
| 2024-10-01|     118962|
| 2024-10-02|     113770|
| 2024-10-03|     108707|
+-----------+-----------+
only showing top 5 rows



In [39]:
trips_by_day.explain(True)

== Parsed Logical Plan ==
'Sort ['pickup_date ASC NULLS FIRST], true
+- Aggregate [pickup_date#504], [pickup_date#504, count(1) AS trips_count#546L]
   +- Filter ((trip_distance#4 > cast(0 as double)) OR (total_amount#16 > cast(0 as double)))
      +- Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, to_date(tpep_pickup_datetime#1, None, Some(Etc/UTC), false) AS pickup_date#504]
         +- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharg

In [40]:
zones = spark.read.option("header", True).option("inferSchema", True).csv("file:///materials/seminar_03_spark/data/raw/taxi_zone_lookup.csv")

In [42]:
zones.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [43]:
zones.count()

265

In [44]:
pickup_zones = zones.select(
    col("LocationID").alias("PULocationID"),
    col("Zone").alias("pickup_zone"),
    col("Borough").alias("pickup_borough")
)

In [46]:
trips_with_zones = clean_trips.join(
    pickup_zones,
    on="PULocationID",
    how="left"
)

In [49]:
trips_with_zones.select("PULocationID", "pickup_zone", "pickup_borough").show(5)

+------------+-------------------+--------------+
|PULocationID|        pickup_zone|pickup_borough|
+------------+-------------------+--------------+
|         162|       Midtown East|     Manhattan|
|          48|       Clinton East|     Manhattan|
|         142|Lincoln Square East|     Manhattan|
|         233|UN/Turtle Bay South|     Manhattan|
|         262|     Yorkville East|     Manhattan|
+------------+-------------------+--------------+
only showing top 5 rows



In [51]:
trips_with_zones.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [PULocationID])
:- Filter ((trip_distance#4 > cast(0 as double)) OR (total_amount#16 > cast(0 as double)))
:  +- Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, to_date(tpep_pickup_datetime#1, None, Some(Etc/UTC), false) AS pickup_date#504]
:     +- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] parquet
+- Project [LocationID#577 AS PULocationID#594, Zone#579 AS 

In [54]:
from pyspark.sql.functions import spark_partition_id
trips = trips.withColumn("partition_id", spark_partition_id())
trips.select("VendorID", "partition_id").show()

+--------+------------+
|VendorID|partition_id|
+--------+------------+
|       2|           1|
|       1|           1|
|       1|           1|
|       1|           1|
|       1|           1|
|       2|           1|
|       1|           1|
|       1|           1|
|       1|           1|
|       1|           1|
|       1|           1|
|       2|           1|
|       2|           1|
|       2|           1|
|       2|           1|
|       1|           1|
|       1|           1|
|       2|           1|
|       2|           1|
|       2|           1|
+--------+------------+
only showing top 20 rows



In [55]:
trips.groupBy(spark_partition_id()).count().show()

+--------------------+-------+
|SPARK_PARTITION_ID()|  count|
+--------------------+-------+
|                   1|1048576|
|                   4|1048576|
|                   7|1048576|
|                  10| 688043|
+--------------------+-------+



In [56]:
trips.repartition(20).groupBy(spark_partition_id()).count().show()

[Stage 46:=======================================>                (14 + 6) / 20]

+--------------------+------+
|SPARK_PARTITION_ID()| count|
+--------------------+------+
|                   0|191687|
|                   1|191688|
|                   2|191688|
|                   3|191688|
|                   4|191689|
|                   5|191689|
|                   6|191689|
|                   7|191689|
|                   8|191689|
|                   9|191689|
|                  10|191689|
|                  11|191689|
|                  12|191689|
|                  13|191688|
|                  14|191689|
|                  15|191689|
|                  16|191689|
|                  17|191688|
|                  18|191688|
|                  19|191688|
+--------------------+------+



In [32]:
clean_trips.repartition("pickup_date").write.mode("overwrite").partitionBy("pickup_date").parquet("file:///spark-local/seminar_03_spark/data/ods/clean_trips")

In [17]:
cd ..

/materials


In [18]:
cd ..

/


In [33]:
ls spark-local/seminar_03_spark/data/ods/clean_trips/pickup_date=2024-10-10/

part-00000-cd467621-9e8c-4b9d-82f2-abc21031d678.c000.snappy.parquet
